# AES 一阶模板攻击（Template Attack）Demo

`ds_atmega2560_aes_power.ets`（Atmega2560 上 AES-128 的功耗迹）的**模板攻击**实现。

## 与 DPA / CPA 的区别

| | DPA / CPA | 模板攻击（本 notebook） |
|---|---|---|
| 前提 | 只有攻击迹（非建模类攻击） | 有同型号设备的**建模迹**（profiling），可用已知密钥标定 |
| 核心模型 | 只用一个标量泄漏模型（如 HW / 单比特）+ 相关系数或差分 | 直接对**多维泄漏分布**建模：每个中间值一个多元高斯模板 $\mathcal{N}(\mu_v,\Sigma_v)$ |
| 判别方式 | 相关/差分峰值 | 最大似然：$\arg\max_k \sum_i \log p(\mathbf{t}_i \mid v=\mathrm{Sbox}[p_i \oplus k])$ |
| 数据效率 | 通常需要更多迹 | 模板建好后，往往几十~几百条迹即可恢复密钥 |

## 流程

1. **建模阶段（profiling）**：把迹按中间值 $v=\mathrm{Sbox}[p\oplus k]$ 分成 256 类，估计每类的均值向量 $\mu_v$ 与协方差矩阵 $\Sigma_v$。
2. **POI 选择**：用 SNR 找出泄漏最强的采样点，只在少量 POI 上建模，避免协方差矩阵维度爆炸。
3. **攻击阶段（attack）**：对每条攻击迹和每个密钥猜测算对数似然，累加后取最大者。

## 本数据集

`ds_atmega2560_aes_power.ets`：**3000 条迹 × 50000 采样点**，明文随机，密钥为 `00 01 02 ... 0f`（便于核对结果）。

因此划分为 **2400 条建模迹 + 600 条攻击迹**。这个比例对模板攻击很关键：
每个子密钥要建 256 个模板，2400/256 ≈ 9.4 条/类，已经接近下限；
若改成 1500 条建模迹（≈5.9 条/类），协方差估计太差，实测只能恢复 7/16 个字节。

In [ ]:
"""一阶模板攻击：常量与工具函数"""

import numpy as np
import matplotlib.pyplot as plt
import estraces

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "PingFang SC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False      # 让负号正常显示

NUM_CLASSES = 256          # 中间值 Sbox[p ^ k] 的取值个数
BYTES = 16                 # AES-128 的 16 个子密钥

# ---------------------------- AES S-BOX ----------------------------
SBOX = np.array((
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16), dtype=np.uint8)


def sub_bytes(plaintext, key_guess):
    """AES 字节替换：Sbox[plaintext ^ key_guess]，结果即为建模用的中间值 v"""
    return SBOX[np.asarray(plaintext, dtype=np.uint8) ^ np.uint8(key_guess)]


def downsample_mean(samples, factor=10):
    """每 factor 个采样点取均值，用于降采样（降低内存与计算量）

    直接在原数组视图上 reshape 后求均值，避免复制整份大矩阵。
    """
    n, p = samples.shape
    return np.asarray(samples).reshape(n, p // factor, factor).mean(axis=2, dtype=np.float64)


def compute_snr(x, labels, n_classes=NUM_CLASSES):
    """逐采样点计算 SNR = Var_v(E[t|v]) / E_v(Var(t|v))

    x      : (n_traces, n_points) 建模迹
    labels : (n_traces,) 每条迹对应的中间值 v ∈ [0, 256)
    返回   : (n_points,) SNR 曲线，值越大说明该点泄漏越强
    """
    x = np.asarray(x, dtype=np.float64)
    n = x.shape[0]
    onehot = np.zeros((n, n_classes), dtype=np.float64)
    onehot[np.arange(n), labels] = 1.0
    counts = np.maximum(onehot.sum(axis=0), 1.0)          # (256,)

    means = (onehot.T @ x) / counts[:, None]              # E[t|v]，(256, n_points)
    sq = (onehot.T @ (x * x)) / counts[:, None]
    var_in = np.maximum(sq - means ** 2, 0.0)             # Var(t|v)

    w = counts / counts.sum()                             # 每类的权重
    mu = (means * w[:, None]).sum(axis=0)                 # 全局均值
    var_between = (((means - mu) ** 2) * w[:, None]).sum(axis=0)
    var_within = (var_in * w[:, None]).sum(axis=0)
    return var_between / (var_within + 1e-12)


def select_poi(snr_curve, n_poi=4, min_gap=5):
    """按 SNR 从高到低挑 POI，并保证相邻 POI 至少相隔 min_gap 个点"""
    pois = []
    for idx in np.argsort(snr_curve)[::-1]:
        if all(abs(int(idx) - q) > min_gap for q in pois):
            pois.append(int(idx))
        if len(pois) == n_poi:
            break
    return np.array(sorted(pois))


def build_templates(x, labels, n_classes=NUM_CLASSES, shrinkage=0.2, min_traces=None):
    """为每个中间值 v 估计一个多元高斯模板 N(mu_v, Sigma_v)

    x        : (n_traces, n_poi) 建模迹在 POI 上的取值
    labels   : (n_traces,) 中间值 v
    shrinkage: 协方差向「池化协方差」收缩的系数（小样本下显著提升稳定性）

    处理小样本的两种手段：
      * 样本充足的类：Sigma_v 向池化协方差 Sigma_pool 收缩
      * 样本不足的类（n_v < min_traces）：直接用 Sigma_pool 作协方差，均值仍用该类的
        （否则该类的对数似然是 -inf，正确密钥会被直接判死）

    返回 dict：
      means / precisions(=Sigma^-1) / consts / valid / counts
      其中 consts[v] = -0.5*log|Sigma_v| - 0.5*mu_v^T Sigma_v^-1 mu_v
    """
    x = np.asarray(x, dtype=np.float64)
    d = x.shape[1]
    if min_traces is None:
        min_traces = d + 2                                # 至少要能估出 d 维协方差

    counts = np.bincount(labels, minlength=n_classes)
    means = np.zeros((n_classes, d))
    covs = [None] * n_classes
    for v in range(n_classes):
        idx = np.flatnonzero(labels == v)
        if idx.size == 0:
            continue
        xv = x[idx]
        means[v] = xv.mean(axis=0)
        if idx.size > 1:
            covs[v] = np.atleast_2d(np.cov(xv, rowvar=False))

    # 池化协方差：各类协方差的加权平均
    acc, wsum = np.zeros((d, d)), 0.0
    for v in range(n_classes):
        if covs[v] is not None:
            acc += covs[v] * counts[v]
            wsum += counts[v]
    pooled = acc / max(wsum, 1.0)
    pooled += np.eye(d) * (1e-9 * np.trace(pooled) / d + 1e-12)

    means_out = np.zeros((n_classes, d))
    precisions = np.zeros((n_classes, d, d))
    consts = np.full(n_classes, -np.inf)
    valid = np.zeros(n_classes, dtype=bool)

    for v in range(n_classes):
        if counts[v] == 0:
            continue                                      # 该中间值从未出现过，无法建模
        own = covs[v] if (counts[v] >= min_traces and covs[v] is not None) else pooled
        cov = (1.0 - shrinkage) * own + shrinkage * pooled
        cov += np.eye(d) * (1e-9 * np.trace(cov) / d + 1e-12)   # 保证正定可逆

        logdet = np.linalg.slogdet(cov)[1]
        prec = np.linalg.inv(cov)
        means_out[v] = means[v]
        precisions[v] = prec
        consts[v] = -0.5 * logdet - 0.5 * means[v] @ prec @ means[v]
        valid[v] = True

    return dict(means=means_out, precisions=precisions, consts=consts,
                valid=valid, counts=counts)


def loglik_matrix(template, x):
    """每条迹在 256 个模板下的对数似然，返回 (n_traces, 256)

    log p(t|v) = -0.5*log|S_v| - 0.5*(t-mu_v)^T S_v^-1 (t-mu_v)
               = consts[v] + t^T S_v^-1 mu_v - 0.5 * t^T S_v^-1 t
    没有建模的类记为 -inf。
    """
    ll = np.full((x.shape[0], NUM_CLASSES), -np.inf)
    for v in np.flatnonzero(template["valid"]):
        prec = template["precisions"][v]
        quad = np.einsum("ij,jk,ik->i", x, prec, x)
        ll[:, v] = template["consts"][v] + x @ (prec @ template["means"][v]) - 0.5 * quad
    return ll


def score_key_guesses(ll, plaintext_col):
    """把逐迹似然按密钥猜测聚合，返回 (最终得分 (256,), 累积得分 (256, n_traces))

    对猜测 k：v_i = Sbox[p_i ^ k]，得分 = sum_i log p(t_i | v_i)
    """
    n = ll.shape[0]
    cols = np.arange(n)[None, :]
    guesses = np.arange(NUM_CLASSES)[:, None]
    cls = sub_bytes(plaintext_col[None, :], guesses)       # (256, n)
    cum = np.cumsum(ll[cols, cls.astype(np.intp)], axis=1)
    return cum[:, -1], cum

In [ ]:
# ------------------------- 1. 加载数据并划分建模集/攻击集 -------------------------
DS_NAME = "ds_atmega2560_aes_power.ets"
NUM_POINTS = 50000         # 每条迹的原始采样点数
COMPRESS = 10              # 降采样倍数
# 本数据集只有 3000 条迹：模板攻击要按 256 个中间值分类，
# 每类至少要有 ~8 条建模迹才稳，所以建模集取 2400 条（平均 9.4 条/类）。
N_PROFILING = 2400         # 建模迹数量
N_ATTACK = 600             # 攻击迹数量（模板建好后并不需要很多迹）
SEED = 0

print(f"加载数据集：{DS_NAME}")
ths = estraces.read_ths_from_ets_file(DS_NAME)
NUM_TRACES = len(ths)

# 降采样：每 COMPRESS 个点取平均值 -> 5000 个点
raw = ths.samples[:NUM_TRACES, :NUM_POINTS]
traces = downsample_mean(raw, COMPRESS)
del raw

plaintext = np.asarray(ths.metadatas["plaintext"][:NUM_TRACES], dtype=np.uint8)
correct_key = np.asarray(ths.metadatas["key"][0], dtype=np.uint8)

# 打乱后切分：模板攻击必须用「两批互不重叠」的迹
perm = np.random.RandomState(SEED).permutation(traces.shape[0])
traces, plaintext = traces[perm], plaintext[perm]

traces_prof, pt_prof = traces[:N_PROFILING], plaintext[:N_PROFILING]
traces_atk, pt_atk = traces[N_PROFILING:N_PROFILING + N_ATTACK], plaintext[N_PROFILING:N_PROFILING + N_ATTACK]

print("总迹数:", traces.shape[0], "| 采样点数:", traces.shape[1])
print("建模迹:", traces_prof.shape, " 攻击迹:", traces_atk.shape)
print("明文:", plaintext.shape, "| 正确密钥:", " ".join(f"{b:02x}" for b in correct_key))

In [ ]:
# ------------------------- 2. POI 选择（基于 SNR） -------------------------
# 说明：建模阶段我们可以拿到建模设备的密钥，因此直接用真实密钥算 SNR 选点，
#       这是模板攻击文献中的标准做法（Chari et al. 2002）。
N_POI = 4          # 每个子密钥使用的 POI 个数（即模板维度）
MIN_GAP = 5        # 相邻 POI 的最小间隔（避免取到同一泄漏尖峰的邻近点）
SHRINKAGE = 0.2    # 协方差收缩系数（向池化协方差收缩）

snr_curves = np.zeros((BYTES, traces_prof.shape[1]))
pois = np.zeros((BYTES, N_POI), dtype=int)

for wi in range(BYTES):
    labels = sub_bytes(pt_prof[:, wi], correct_key[wi])
    snr_curves[wi] = compute_snr(traces_prof, labels)
    pois[wi] = select_poi(snr_curves[wi], N_POI, MIN_GAP)

print("每个字节选出的 POI：")
for wi in range(BYTES):
    print(f"  byte {wi:2d}: {pois[wi]}  (SNR = {np.round(snr_curves[wi][pois[wi]], 4)})")

# 可视化：第 0 个字节的 SNR 曲线 + POI 位置
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)
axes[0].plot(traces_prof.mean(axis=0), color="0.6", lw=0.8, label="平均迹")
axes[0].set_title("平均功耗迹（降采样后）")
axes[0].set_ylabel("功耗")

axes[1].plot(snr_curves[0], color="tab:blue", lw=0.9, label="SNR")
axes[1].plot(pois[0], snr_curves[0][pois[0]], "rv", ms=10, label="选中的 POI")
axes[1].set_title("Byte 0 的 SNR 曲线（峰值即泄漏点）")
axes[1].set_xlabel("采样点")
axes[1].set_ylabel("SNR")
axes[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
# ------------------------- 3. 建模：为 16 个字节各建一套模板 -------------------------
templates = []
for wi in range(BYTES):
    labels = sub_bytes(pt_prof[:, wi], correct_key[wi])
    x = traces_prof[:, pois[wi]]                      # (n_prof, N_POI)
    tpl = build_templates(x, labels, shrinkage=SHRINKAGE)
    templates.append(tpl)
    avg = np.bincount(labels, minlength=NUM_CLASSES).mean()
    print(f"byte {wi:2d}: 有效模板 {tpl['valid'].sum():3d}/256, 每类平均 {avg:.1f} 条建模迹")

# 各类建模迹条数分布
counts0 = np.bincount(sub_bytes(pt_prof[:, 0], correct_key[0]), minlength=NUM_CLASSES)
plt.figure(figsize=(9, 3))
plt.bar(np.arange(NUM_CLASSES), counts0, width=1.0, color="tab:blue")
plt.title("Byte 0：每个中间值 v 对应的建模迹条数")
plt.xlabel("中间值 v = Sbox[p ^ k]")
plt.ylabel("建模迹条数")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------- 4. 攻击：最大似然估计密钥 -------------------------
recovered = np.zeros(BYTES, dtype=int)
ranks = np.zeros(BYTES, dtype=int)          # 正确密钥在得分排序中的名次（0 = 第一）
min_traces = np.zeros(BYTES, dtype=int)     # 累积多少条迹后首次命中正确密钥
best_over_time = np.zeros((BYTES, N_ATTACK), dtype=int)
scores_all = np.zeros((BYTES, NUM_CLASSES))

for wi in range(BYTES):
    x = traces_atk[:, pois[wi]].astype(np.float64)
    ll = loglik_matrix(templates[wi], x)                 # (n_atk, 256)
    scores, cum = score_key_guesses(ll, pt_atk[:, wi])    # (256,), (256, n_atk)
    scores_all[wi] = scores

    recovered[wi] = int(np.argmax(scores))
    ranks[wi] = int((scores > scores[correct_key[wi]]).sum())

    best = np.argmax(cum, axis=0)
    best_over_time[wi] = best
    hit = np.flatnonzero(best == correct_key[wi])
    min_traces[wi] = int(hit[0] + 1) if hit.size else -1
    print(f"byte {wi:2d}: recovered = 0x{recovered[wi]:02x}, rank = {ranks[wi]:3d}")

# 每个字节的得分曲线（红色竖线 = 正确密钥）—— 对应 DPA 里的差分尖峰图
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(14, 9))
for wi in range(BYTES):
    ax = axes[wi // 4, wi % 4]
    s = scores_all[wi] - np.max(scores_all[wi])
    ax.plot(s, lw=0.8, color="tab:blue")
    ax.axvline(int(correct_key[wi]), color="red", ls="--", lw=1.2)
    ax.set_title(f"byte {wi}: rank {ranks[wi]}", fontsize=9)
    ax.set_xlim(0, NUM_CLASSES - 1)
    ax.tick_params(labelsize=7)
fig.suptitle("各字节的对数似然得分（相对最大值），红色虚线 = 正确密钥", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# ------------------------- 5. 结果汇总与可视化 -------------------------
def display_key_analysis(recovered_keys, correct_keys):
    """对比并展示恢复的密钥与正确密钥的差异。"""
    print("\n" + "=" * 65)
    print(f"{'密钥字节分析 (Key Byte Analysis)':^60}")
    print("=" * 65)
    print(f"{'Idx':<6} | {'Recovered':<12} | {'Status':<8} | {'Correct':<12}")
    print("-" * 65)
    correct_count = 0
    for i in range(len(correct_keys)):
        rec_byte = recovered_keys[i] if i < len(recovered_keys) else None
        cor_byte = correct_keys[i]
        rec_str = f"0x{rec_byte:02x}" if rec_byte is not None else "None"
        cor_str = f"0x{cor_byte:02x}"
        if rec_byte == cor_byte:
            status = "✅ Match"
            correct_count += 1
        else:
            status = "❌ Error"
        print(f"{i:<6} | {rec_str:<12} | {status:<8} | {cor_str:<12}")
    print("-" * 65)
    total = len(correct_keys)
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    print(f"\n📊 统计结果:")
    print(f"   总字节数: {total}")
    print(f"   正确匹配: {correct_count}")
    print(f"   攻击准确率: {accuracy:.1f}%")
    if accuracy == 100.0:
        print("   🎉 恭喜！所有子密钥均已成功恢复！")
    else:
        print("   ⚠️ 仍有部分密钥未恢复或错误。")
    print("=" * 65 + "\n")


print("The recovered Key is: ")
for b in recovered:
    print("%02x " % b, end="")
print("\nThe correct Key is: ")
for b in correct_key:
    print("%02x " % b, end="")
print()

display_key_analysis(recovered, correct_key)

# 需要多少条攻击迹才能恢复全部子密钥？
n_recovered = (best_over_time == correct_key[:, None]).sum(axis=0)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, N_ATTACK + 1), n_recovered, color="tab:red", lw=2)
plt.axhline(BYTES, color="0.5", ls="--", lw=1, label="16 个子密钥全部恢复")
plt.xlabel("使用的攻击迹条数")
plt.ylabel("已恢复的子密钥个数")
plt.title("模板攻击的密钥恢复曲线（success rate vs. #traces）")
plt.ylim(-0.5, BYTES + 0.5)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("每个字节所需迹数（首次命中正确密钥）：", min_traces)
print("每个字节的密钥排名（0 = 排名第一）：", ranks)

## 备注：从 DPA 改到 TA 的对应关系与调参提示

**与原 DPA 版本的对应关系**

| 原 DPA 代码 | 本 TA 版本 |
|---|---|
| 对每个猜测 $k$ 按 Sbox 输出的某一比特把迹分成两组，求均值差 | 按 256 个中间值 $v=\mathrm{Sbox}[p\oplus k]$ 把**建模迹**分成 256 组，每组估计 $(\mu_v, \Sigma_v)$ |
| `dpa_max_diff[kg] = max(|mean1 - mean0|)` | `scores[kg] = Σ_i log N(t_i; μ_{v_i}, Σ_{v_i})` |
| `np.argmax(dpa_max_diff)` | 同样 `np.argmax(scores)`，但判据是似然而非差分幅度 |

**调参提示**

- `N_POI`（模板维度）：2~6 之间权衡。维度越高信息越多，但协方差估计需要更多建模迹。本数据集上 2~4 个 POI 结果几乎相同（泄漏集中在同一尖峰附近）。
- `SHRINKAGE`：协方差向池化协方差收缩的比例，0.1~0.5。样本越少应越大。
- 若某些字节 `rank != 0`：优先增大 `N_PROFILING`，其次减小 `N_POI` 或增大 `SHRINKAGE`。
- 模板可以**复用**：同一设备换了密钥后，建模迹建立的模板依然有效，只需更换攻击迹（这也是模板攻击对实现者的最大威胁）。
- 想看"模板长什么样"，可以打印 `templates[0]["means"]`（256×N_POI 的均值矩阵）观察不同中间值的泄漏差异。